# 여행 상품 구매 예측
**목표**: 고객 정보로 여행 상품을 살지 안 살지 예측하기  

데이터 셋 :
https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction

**데이터**: Travel.csv (4,888명, 변수 20개)




팀원: 김준수 · 김한별 · 서동열
---
```
1단계. 패키지 설치 & 데이터 불러오기
2단계. 데이터 살펴보기 (EDA)
3단계. 전처리 및 피처 엔지니어링
4단계. 3가지 모델 학습 & 성능 비교
5단계. 하이퍼파라미터 튜닝
6단계. 혼동행렬(Confusion Matrix)
7단계. 최종 모델 변수 중요도 분석
8단계. 모델 저장

```

## 1단계. 패키지 설치 & 데이터 불러오기


In [5]:
# 라이브러리 설치 (LGBM, plotly)
!pip install -q lightgbm plotly

# 데이터 처리
import pandas as pd, numpy as np, pickle, warnings

# 불필요한 경고 숨기기
warnings.filterwarnings('ignore')


# 라이브러리 호출

# 로지스틱 회귀 : 기본 분류 모델
from sklearn.linear_model import LogisticRegression
# 랜덤포레스트 : 여러개의 트리를 사용하는 앙상블 모델
from sklearn.ensemble import RandomForestClassifier
# LightGBM : 부스팅 계열 분류 모델
from lightgbm import LGBMClassifier

# 평가 성능 지표
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# 데이터 분할
from sklearn.model_selection import train_test_split

# 시각화
import plotly.express as px
import plotly.graph_objects as go

# 전처리 파이프라인 구성
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 결측치 처리 / 인코딩 / 스케일링
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# CSV 파일 불러오기
df = pd.read_csv('/content/drive/MyDrive/Travel.csv')

# 데이터 크기와 샘플 확인
print(f'데이터 크기: {df.shape[0]}명 x {df.shape[1]}개 변수')
display(df.head(3))  # 상위 3개 행 확인

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
데이터 크기: 4888명 x 20개 변수


,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0


---
## 2단계. 데이터 살펴보기 (EDA)

결측치, 구매 비율, 직업별 구매 패턴을 시각화합니다.

In [6]:
# 결측치 개수 확인
# 비어있는 부분이 있다면 , 에러가 날 수 있다.
# 나이가 비어있다 던지 (NaN등..) 비어있는 값을 채워주기 위해 살펴보는 과정
missing = df.isnull().sum() # 각 컬럼별 결측치를 계산한다.
missing = missing[missing > 0].sort_values(ascending=True)

# ======== 시각화 ========
# 막대형 그래프 : 컬럼별 결측치 개수
px.bar(x=missing.values, y=missing.index, orientation='h',
       title='결측치 개수', text=missing.values,
       color=missing.values, color_continuous_scale='Reds',
       labels={'x':'결측치 수','y':'컬럼명'}).show()

# 구매 여부 분포 확인 하기
# 타깃 값이 한쪽으로 치우쳐진건 아닌지 확인하기 위해 살펴보는 과정
cnt = df['ProdTaken'].value_counts().rename(index={0:'미구매(0)',1:'구매(1)'})


# 시각화 (파이 차트) : 구매 여부 분포
fig2 = px.pie(names=cnt.index, values=cnt.values, title='구매 여부 분포',
              color_discrete_sequence=['lightblue','salmon'], hole=0.3)
fig2.update_traces(textinfo='label+percent+value')
fig2.show()


# 직업별 구매율을 살펴본다. (중요한 피쳐인지 아닌지 확인한다.)
# 막대형 그래프 : 직업별 구매율

occ = df.groupby('Occupation')['ProdTaken'].mean().sort_values().reset_index()
occ.columns = ['직업','구매율']


fig3 = px.bar(occ, x='구매율', y='직업', orientation='h',
              title='직업별 구매율',
              text=[f'{v:.1%}' for v in occ['구매율']],
              color_discrete_sequence=['royalblue'])
fig3.update_traces(textposition='outside')
fig3.show()

In [7]:
occ_check = df.groupby('Occupation')['ProdTaken'].agg(['mean', 'count', 'sum']).sort_values('mean', ascending=False)
occ_check.columns = ['구매율', '인원수', '구매자수']
display(occ_check)

,구매율,인원수,구매자수
Occupation,,,
Free Lancer,1.000000,2,2
Large Business,0.276498,434,120
Small Business,0.184261,2084,384
Salaried,0.174831,2368,414


Free Lancer의 구매율이 100%로 나타났지만, 이는 해당 직업군의 표본 수가 2명으로 매우 적고 그 2명이 모두 구매했기 때문에 나온 결과

---
## 3단계. 전처리 및 피처 엔지니어링

**새 변수 추가**: `TotalVisitors`(총 방문 인원), `HighIncome`(고소득 여부)

| 컬럼 타입 | 결측치 처리 | 변환 |
|-----------|------------|------|
| 숫자형 | 중앙값으로 대체 | 표준화 |
| 범주형(문자) | 최빈값으로 대체 | 원-핫 인코딩 |

In [8]:
# 파생 변수 추가 (원래 없던 컬럼)

# 총 방문 인원 추가
df['NumberOfChildrenVisiting'] = df['NumberOfChildrenVisiting'].fillna(0) # 동반 자녀 수가 비어 있으면 0으로 채움
df['TotalVisitors'] = df['NumberOfPersonVisiting'] + df['NumberOfChildrenVisiting'] # 총 방문 인원 수를 보기 위해 새 변수를 만듦 (방문인원 + 동반 자녀수)
# 월소득이 중앙값 이상이라면 1 , 아니라면 0
# 중앙값 ex.) 1 ~ 1000 중앙값 은 500.5

# 고소득 여부 변수 추가
df['HighIncome']    = (df['MonthlyIncome'] >= df['MonthlyIncome'].median()).astype(int)

# 문제 x 와 / 정답(label) y 를 나눔 .
X = df.drop(columns=['CustomerID','ProdTaken']) # 고객 ID와 구매여부 를 제외
# 이유 : 고객 아이디는 예측에 도움이 되지 않아서, 구매여부는 정답(y)이기 때문에 입력에서 제외한다.

y = df['ProdTaken'] # 구매 여부를 정답(y)으로 설정

# 데이터를 학습용(80%)과 테스트용(20%)으로 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'학습: {X_train.shape[0]}명 / 테스트: {X_test.shape[0]}명')

# 숫자형 컬럼과 범주형 컬럼 분리
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()

# 숫자형 / 범주형은 전처리 방법이다르다.
# 숫자데이터는 표준화
# 문자데이터는 원핫 인코딩 방식을 사용하여야 한다.

# 숫자형: 빈 값 -> 중앙값 -> 표준화
num_pipe = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
# 범주형: 빈 값 -> 최빈값 -> 원-핫 인코딩(문자를 0/1로)
cat_pipe = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])

# 전처리 된 숫자형, 범주형 데이터를 하나로 합침.
preprocessor = ColumnTransformer([('num',num_pipe,num_cols),('cat',cat_pipe,cat_cols)])
print('전처리 준비 됨')

학습: 3910명 / 테스트: 978명
전처리 준비 됨


---
## 4단계. 3가지 모델 학습 & 성능 비교

전처리는 공통으로 사용하고 , 모델만 교체 했습니다.

| 모델 | 특징 |
|------|------|
| **로지스틱 회귀** | 기준선 역할 |
| **랜덤 포레스트** | 안정적 |
| **LightGBM** | 매우 빠르고 정확 |

추가로 XGBoost도 테스트했지만,  
비교 구조를 단순하게 정리하기 위해 최종 리팩토링 과정에서는 제외했습니다.

In [9]:
# 비교할 모델을 설정
models_config = {
    # 구매/ 미구매 를 분류하는 이진 분류 모델 .최대 반복횟수를 1000으로 설정. 결과재현.
    '로지스틱 회귀': LogisticRegression(max_iter=1000, random_state=42),


    # 트리의 개수를 200개 로 설정, 많으면 안정적일 수 있으나 느려짐.
    # class_weight='balanced= 클래스의 불균형을 자동 조절해주는 옵션
    '랜덤 포레스트': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),

    # 부스팅 단계 400개, 학습률 0.05 로 지정 .(보폭 개념, 크면 많이 움직임. 크면 불안정. 작으면 오래걸림, 조금씩 조정. )
    # verbose = 학습하면서 진행상황, 경고, 로그 등을 보여줌. 너무 많이 보여주면 번잡하니 로그 출력을 최소화 함
    'LightGBM':      LGBMClassifier(n_estimators=400, learning_rate=0.05, class_weight='balanced', random_state=42, verbose=-1),
}
# 결과를 저장할 리스트와, 학습된 내용을 저장할 딕셔너리를 만듦.
results, trained_pipes = [], {}

# 모델별로 학습/ 예측/ 평가 진행
for name, model in models_config.items():
  # 전처리와 모델을 하나의 파이프라인으로 묶음
    pipe = Pipeline([('preprocessor',preprocessor),('model',model)])
    # 학습 데이터로 모델 학습
    pipe.fit(X_train, y_train)

    # 학습 데이터로 모델 학습
    y_pred  = pipe.predict(X_test)

    # 테스트 데이터 예측값(0/1)
    y_proba = pipe.predict_proba(X_test)[:,1]

    #============== # 모델별 평가지표 계산 ==================

    # r = 지표 계산(성적표) =
    # {
    #     정확도 ) Accuracy :  전체 중 맞춘 비율 ,
    #     정밀도 ) Precision : 구매라고 생각한 것 중 실제 구매한 비율,
    #     재현율 ) Recall ; 실제 구매자중 찾아낸 비율 ,
    #     F1 점수) F1 score : precions, recall 의 균형 점수
    #     ROC- AOC  : 전반적인 분류 성능
    # }

    r = {'모델':name,
         'Accuracy':round(accuracy_score(y_test,y_pred),4),
         'Precision':round(precision_score(y_test,y_pred,zero_division=0),4),
         'Recall':round(recall_score(y_test,y_pred,zero_division=0),4),
         'F1 Score':round(f1_score(y_test,y_pred,zero_division=0),4),
         'ROC-AUC':round(roc_auc_score(y_test,y_proba),4)}

    # ==========================================================
    # 결과 리스트에 저장
    results.append(r)
    # 학습된 파이프라인을 저장
    trained_pipes[name] = pipe

    # 모델별 주요 성능 (AUC, F1 score ) 출력
    print(f'  {name:<15} | AUC={r["ROC-AUC"]:.4f}  F1={r["F1 Score"]:.4f}')
# 결과를 데이터 프레임으로 정리하고,ROC-AUC 기준으로 내림차순으로 정렬
results_df = pd.DataFrame(results).sort_values('ROC-AUC',ascending=False).reset_index(drop=True)
# 인덱스가 0부터 시작하므로.. 1부터 시작하게 만들기 위해 설정
results_df.index += 1
print('모든 모델 학습 완료!')
display(results_df)

  로지스틱 회귀         | AUC=0.8008  F1=0.4229
  랜덤 포레스트         | AUC=0.9702  F1=0.6411
  LightGBM        | AUC=0.9580  F1=0.8398
모든 모델 학습 완료!


,모델,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,랜덤 포레스트,0.8947,0.9583,0.4817,0.6411,0.9702
2,LightGBM,0.9407,0.8889,0.7958,0.8398,0.9580
3,로지스틱 회귀,0.8354,0.6705,0.3089,0.4229,0.8008


In [10]:
# 모델별 성능 지표를 그래프용 데이터 형태로 변환
df_melt = results_df.melt(id_vars='모델', value_vars=['Accuracy','F1 Score','ROC-AUC'],
                          var_name='지표', value_name='점수')

# 모델별 Accuracy, F1 score , ROC - AUC 를 비교하는 그룹 막대그래프 생성
fig = px.bar(df_melt, x='모델', y='점수', color='지표',
             barmode='group', title='3가지 모델 성능 비교',
             text='점수', height=420)

# 막대 위에 점수를 소수점 3자리 까지 표시
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')

# y축 범위를 조정해 텍스트가 잘리지 않도록 설정
fig.update_layout(yaxis_range=[0,1.15])


fig.show()


이 그래프는 세 가지 모델의 성능 비교 결과입니다.
파란색은 정확도, 주황색은 F1 점수, 초록색은 ROC-AUC입니다.
LightGBM은 정확도와 F1 점수가 가장 높았습니다.
랜덤 포레스트는 ROC-AUC가 가장 높았습니다.
로지스틱 회귀는 전체적으로 성능이 가장 낮았습니다.
그래서 이번 데이터에서는 LightGBM이 가장 균형 잡힌 모델이라고 판단했습니다.

In [11]:
# ROC 곡선
fig_roc = go.Figure()

# 학습된 모델별 ROC 곡선 계산 및 추가
for name, pipe in trained_pipes.items():
    prob = pipe.predict_proba(X_test)[:,1]  # 구매(1) 일 확률 예측
    fpr, tpr, _ = roc_curve(y_test, prob) # FPR(오탐률), TPR (재현율) 계산
    auc = roc_auc_score(y_test, prob) # ROC-AUC 점수 계산

    # 모델별 ROC 곡선 추가
    fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={auc:.3f})'))

# 랜덤 기준선 추가
fig_roc.add_trace(go.Scatter(x=[0,1],y=[0,1],mode='lines',
    name='랜덤(AUC=0.5)',line=dict(color='gray',dash='dash')))

# 그래프 제목 및 축이름 설정
fig_roc.update_layout(title='ROC 곡선 비교 (왼쪽 위에 가까울수록 좋음)',
    xaxis_title='오탐률(FPR)', yaxis_title='재현율(TPR)', height=420)

fig_roc.show()

ROC 곡선은 모델의 전반적인 분류 성능을 비교하는 그래프입니다.
그래프가 왼쪽 위에 가까울수록 좋은 성능을 의미하며, 회색 점선은 랜덤 예측 기준선입니다.
세 모델 모두 랜덤 기준선보다 높은 성능을 보였고, 그중 랜덤 포레스트가 AUC 0.970으로 가장 높은 값을 나타냈습니다.

정리 :

ROC 곡선에서도 세 모델 모두 랜덤 기준선보다 높은 성능을 보였으며,


랜덤 포레스트가 가장 높은 AUC 값을 나타냈습니다.


다만 정확도(Accuracy)와 F1Score(정밀도와 재현율의 균형)까지


함께 고려했을 때 LightGBM이 가장 균형 잡힌 성능을 보여 최종 모델로 선택했습니다.



## 5단계. 하이퍼파라미터 튜닝
기본 모델 비교 결과를 바탕으로,
성능이 좋았던 LightGBM 모델에 대해 하이퍼파라미터 튜닝을 진행하였습니다.


RandomizedSearchCV를 사용하여

주요 파라미터를 탐색하고, ROC-AUC 기준으로 최적 모델을 선택하였습니다.

In [12]:
# LGBM 하이퍼파라미터 튜닝
# RandomizedSearchCV는 여러 하이퍼파라미터 조합을 랜덤하게 시험해보고,
# 교차검증을 통해 가장 좋은 조합을 찾는 방법이다.
# (GridSearchCV는 모든 조합을 다 시도하고, RandomizedSearchCV는 일부 조합만 랜덤하게 시도한다.)

from sklearn.model_selection import RandomizedSearchCV

# 탐색할 하이퍼파라미터 후보 설정
param_dist = {
    'model__n_estimators': [200, 300, 400],      # 트리개수
    'model__learning_rate': [0.03, 0.05, 0.1],   # 학습률
    'model__num_leaves': [15, 31, 63]            # 트리 가지수
}

# 전처리 + LightGBM 모델 파이프라인 구성
lgb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    # 클래스 불균형 보정, 결과재현 , 로그메세지 최소화
    ('model', LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1))
])

# RandomizedSearchCV로 최적의 파라미터 탐색
search = RandomizedSearchCV(
    lgb_pipe,
    param_distributions=param_dist,
    n_iter=5,            # 랜덤 조합을 5번 시험
    scoring='roc_auc',   # ROC-AUC 기준으로 평가
    cv=3,                # 데이터를 3부분으로 나눠 3번 검증 (3-Fold)
    random_state=42,     # 결과 재현

)

# 학습 데이터로 튜닝 진행
search.fit(X_train, y_train)

# 최적의 파라미터와 점수 출력
print("최적 파라미터:", search.best_params_)                # 가장 성능이 좋았던 설정값
print("최적 CV ROC-AUC:", round(search.best_score_, 4))     # 가장 좋았던 교차검증 ROC-AUC 점수

# 튜닝된 LightGBM을 최종 모델로 저장
best_name = 'LightGBM (Tuned)'
best_pipe = search.best_estimator_

# 튜닝된 모델로 테스트 데이터 예측 해보기
y_pred_tuned = best_pipe.predict(X_test)
y_proba_tuned = best_pipe.predict_proba(X_test)[:,1]

최적 파라미터: {'model__num_leaves': 63, 'model__n_estimators': 400, 'model__learning_rate': 0.03}
최적 CV ROC-AUC: 0.9291


In [13]:
# 튜닝된 LightGBM 성능을 데이터프레임으로 정리
tuned_result = pd.DataFrame([{
    '모델': 'LightGBM (Tuned)',
    'Accuracy': round(accuracy_score(y_test, y_pred_tuned), 4),
    'Precision': round(precision_score(y_test, y_pred_tuned, zero_division=0), 4),
    'Recall': round(recall_score(y_test, y_pred_tuned, zero_division=0), 4),
    'F1 Score': round(f1_score(y_test, y_pred_tuned, zero_division=0), 4),
    'ROC-AUC': round(roc_auc_score(y_test, y_proba_tuned), 4)
}])
# 튜닝된 결과 시각화
display(tuned_result)
print()
# 이전 모델들과 비교
compare_df = pd.concat([results_df, tuned_result], ignore_index=True)
display(compare_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True))

,모델,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LightGBM (Tuned),0.9519,0.9235,0.822,0.8698,0.9645


,모델,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,랜덤 포레스트,0.8947,0.9583,0.4817,0.6411,0.9702
1,LightGBM (Tuned),0.9519,0.9235,0.8220,0.8698,0.9645
2,LightGBM,0.9407,0.8889,0.7958,0.8398,0.9580
3,로지스틱 회귀,0.8354,0.6705,0.3089,0.4229,0.8008


비교 결과, 랜덤 포레스트는 ROC-AUC가 가장 높았지만

Recall이 낮아 실제 구매 고객을 놓치는 비율이 높았습니다.


반면 튜닝된 LightGBM은 Accuracy, F1 Score, Recall이 모두 높게 나타나

전반적으로 가장 균형 잡힌 성능을 보였습니다.


따라서 최종 모델은 튜닝된 LightGBM으로 선택했습니다.

In [14]:
# 기본 LightGBM과 튜닝된 LightGBM만 비교
lgb_compare = compare_df[compare_df['모델'].isin(['LightGBM', 'LightGBM (Tuned)'])]

lgb_melt = lgb_compare.melt(
    id_vars='모델',
    value_vars=['Accuracy', 'F1 Score', 'ROC-AUC'],
    var_name='지표',
    value_name='점수'
)

fig_lgb = px.bar(
    lgb_melt,
    x='모델',
    y='점수',
    color='지표',
    barmode='group',
    title='LightGBM 튜닝 전/후 성능 비교',
    text='점수',
    height=420
)

fig_lgb.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_lgb.update_layout(yaxis_range=[0, 1.15])
fig_lgb.show()

### 6단계. 혼동행렬(Confusion Matrix)
혼동행렬은 실제 값과 예측 값을 비교해서, 어떤 경우를 맞췄고 어떤 경우를 틀렸는지 보여주는 표입니다.

- TN: 미구매 고객을 정확히 미구매로 예측
- FP: 미구매 고객인데 구매로 잘못 예측
- FN: 실제 구매 고객을 놓침
- TP: 구매 고객을 정확히 구매로 예측

In [15]:
# 선택한 모델로 예측값 / 예측 확률 생성
y_pred_b = best_pipe.predict(X_test)
y_proba_b = best_pipe.predict_proba(X_test)[:, 1]

# 최종 모델 주요 성능 출력
print(f'최종 선택 모델: {best_name}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_b):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_b, zero_division=0):.4f}')

# 혼동행렬 계산
cm = confusion_matrix(y_test, y_pred_b)
tn, fp, fn, tp = cm.ravel()

# 혼동행렬 시각화
px.imshow(
    cm,
    labels=dict(x='예측', y='실제', color='고객 수'),
    x=['예측: 미구매(0)', '예측: 구매(1)'],
    y=['실제: 미구매(0)', '실제: 구매(1)'],
    title=f'혼동 행렬 - {best_name}',
    color_continuous_scale='Blues',
    text_auto=True
).update_layout(height=360).show()

# 혼동행렬 해석
print(f'TN={tn}: 미구매 -> 정확히 예측')
print(f'FP={fp}: 미구매인데 구매로 잘못 예측')
print(f'FN={fn}: 구매할 고객을 미구매로 놓침')
print(f'TP={tp}: 구매 -> 정확히 예측')

최종 선택 모델: LightGBM (Tuned)
ROC-AUC: 0.9645
F1 Score: 0.8698


TN=774: 미구매 -> 정확히 예측
FP=13: 미구매인데 구매로 잘못 예측
FN=34: 구매할 고객을 미구매로 놓침
TP=157: 구매 -> 정확히 예측


## 7단계. 최종 모델 변수 중요도 분석

In [16]:

# 최종모델 , 전처리기 불러오기
clf = best_pipe.named_steps['model']
pre = best_pipe.named_steps['preprocessor']

# 전처리 후 변수 이름 가져오기
feat_names = pre.get_feature_names_out()

# 변수 중요도 데이터 프레임 만들기
imp_df = pd.DataFrame({'변수':feat_names,'중요도':clf.feature_importances_})
# 변수 이름 정리
imp_df['변수'] = imp_df['변수'].str.replace('num__','').str.replace('cat__','')
# 중요도가 높은 변수 상위 15개 선택
imp_df = imp_df.nlargest(15,'중요도').sort_values('중요도')

# 변수 중요도 시각화

px.bar(imp_df, x='중요도', y='변수', orientation='h',
       title=f'변수 중요도 Top 15 ({best_name})',
       text='중요도', color='중요도', color_continuous_scale='Viridis'
).update_traces(texttemplate='%{text:.4f}',textposition='outside'
).update_layout(height=480).show()

# 가장 중요한 변수 확인
top_var = imp_df.iloc[-1]['변수']
print(f'예측에 가장 큰 영향을 준 변수: {top_var}')

예측에 가장 큰 영향을 준 변수: MonthlyIncome


이 그래프는 최종 선택한 모델인 LightGBM이

중요하게 판단한 변수 상위 15개를 보여줍니다.


중요도가 높을수록 예측에 더 큰 영향을 준 변수라고 해석할 수 있습니다.


결과에서는 MonthlyIncome(월소득),DurationOfPitch(고객에게 상품을 설명한 시간),Age가

특히 중요한 변수로 나타났으며, 그중 MonthlyIncome이 가장 큰 영향을 준 변수로 확인되었습니다.

## 8단계. 모델 저장
학습한 최종 모델을 파일로 저장해두면, 다음에 다시 학습하지 않고 바로 불러와 사용할 수 있습니다.

In [17]:
# 최종 모델 저장 경로 설정
save_path = 'travel_best_model.pkl'
# 학습한 모델 저장
with open(save_path,'wb') as f:
    pickle.dump(best_pipe, f)
print(f'저장 완료: {save_path}')

# 저장한 모델 다시 불러오기
with open(save_path,'rb') as f:
    loaded_model = pickle.load(f)
print('불러오기 완료')

# 불러온 모델의 성능 확인
auc_check = roc_auc_score(y_test, loaded_model.predict_proba(X_test)[:,1])
print(f'ROC-AUC: {auc_check:.4f} (<- 위에서 나온 결과와 동일하면 정상)')

저장 완료: travel_best_model.pkl
불러오기 완료
ROC-AUC: 0.9645 (<- 위에서 나온 결과와 동일하면 정상)
